# HW02 — Store the events in a private S3 data lake

Create the assignment bucket with Boto3, then upload the hourly extracts under `wikipedia-hourly/`. Authentication uses the EC2 environment's AWS credentials; no keys are stored in this notebook.


In [1]:
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

NETID = "bz263"
BUCKET = f"dsan6000-{NETID}"
PREFIX = "wikipedia-hourly/"
REGION = "us-east-1"
DATA_DIR = Path("data")

session = boto3.Session(region_name=REGION)
s3 = session.client("s3")
account_id = session.client("sts").get_caller_identity()["Account"]

In [2]:
try:
    s3.head_bucket(Bucket=BUCKET, ExpectedBucketOwner=account_id)
    print(f"Using existing bucket: {BUCKET}")
except ClientError as exc:
    if exc.response["Error"]["Code"] not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    # us-east-1 does not take a LocationConstraint.
    s3.create_bucket(Bucket=BUCKET)
    s3.get_waiter("bucket_exists").wait(Bucket=BUCKET)
    print(f"Created bucket: {BUCKET}")

s3.put_public_access_block(
    Bucket=BUCKET,
    ExpectedBucketOwner=account_id,
    PublicAccessBlockConfiguration={
        "BlockPublicAcls": True,
        "IgnorePublicAcls": True,
        "BlockPublicPolicy": True,
        "RestrictPublicBuckets": True,
    },
)
print("All four S3 Block Public Access settings are enabled.")

Created bucket: dsan6000-bz263
All four S3 Block Public Access settings are enabled.


In [3]:
files = sorted(DATA_DIR.glob("*.parquet"))
assert len(files) == 24, f"Expected 24 local hourly files, found {len(files)}. Run notebook 1 first."

for path in files:
    key = PREFIX + path.name
    s3.upload_file(str(path), BUCKET, key)
    metadata = s3.head_object(Bucket=BUCKET, Key=key, ExpectedBucketOwner=account_id)
    assert metadata["ContentLength"] == path.stat().st_size, f"Size mismatch: {key}"

pages = s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX)
uploaded = [obj for page in pages for obj in page.get("Contents", [])
            if obj["Key"].endswith(".parquet")]
assert {obj["Key"] for obj in uploaded} == {PREFIX + path.name for path in files}
print(f"Verified {len(uploaded)} files in s3://{BUCKET}/{PREFIX}")
print(f"Total size: {sum(obj['Size'] for obj in uploaded):,} bytes")

Verified 24 files in s3://dsan6000-bz263/wikipedia-hourly/
Total size: 32,012,082 bytes
